---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-20: A Deep Dive into Transformer Architecture - II</h1>

# Learning agenda of this notebook
1. Recap of Previous Session
2. Why we need Self-Attention?
3. What is Self-Attention?
4. . How Self-Attention Work?
5. What is Masked (Causal) Self Attention?
6. What is Cross Self Attention?
7. What is Multi-Headed Self Attention?
8. Layer Normalization, Residual Connections and Feed Forward Neural Network
9. From 6th Decoder Output to Token Prediction
10. Visualization of Transformer Architecture
11. Example Questions

# <span style='background :lightgreen' >1. Recap</span>

<h2 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The <b>Transformer architecture</b>  was introduced  by Google researchers headed by Ashish Vaswani published in a paper titled <b>Attention Is All You Need</b> in 2017.</h2>



<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The <b>Transformer Architecture</b> has gone beyond NLP and text-based tasks and there exists Vision based Transformer architectures that have done really impressive tasks in Computer Vision and Audio Based Transformer architectures.</h3>

<div style="text-align:center;">
    <img src="../images/tfmr-007.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>



# <span style='background :lightgreen' >2. Why we need Self-Attention?</span>

**Example 1:** Consider the following two sentences, that demonstrate **Co-reference Resolution**:
```
“The animal didn't cross the street because it was too tired."
“The animal didn't cross the street because it was too wide."
```
- How does a model know that “it” refers to  the word "animal" in the first sentence and to the word "street" in the second sentence?
    - In the first sentence, "it" refers to "animal", because animals get tired — streets do not get tired
    - In the second sentence, "it" refers to "street", because streets can be wide — animals are not described as wide in this context


**Example 2:** Consider the following two sentences, that demonstrate **Word Sense Disambiguation**:
```
“Money bank grows."
“River bank flows."
```
- How does a model know that “bank” refers to  a financial institute in the first sentence and refers to the land alongside a water channel in the second sentence?
    - In the first sentence, "bank" refers to a financial institution, because the word "money" is mostly used in financial context
    - In the second sentence, "bank" refers to a river bank, because the word "river" is mostly used in geographical context

**Example 3:** Consider the following two sentences, having word sense disambiguation, coreference resolution and case sensitivity as a bonus issue:
```
“She ate an apple after lunch because it was fresh and sweet."
“She bought an apple because her old phone stopped working."
```
- How does a model know that “apple” refers to  fruit in the first sentence and refers to company in the second sentence?
    - In the first sentence, "apple" refers to a fruit, because of the contextual clues like "ate", "lunch", "fresh" and "sweet"
    - In the second sentence, "apple" refers to a company, because of the contextual clues like "bought", "phone" and "stopped working"
    - The word "apple" in the second sentence is in lower case, but, the contextual clues alone would correctly disambiguate it to company and not fruit

# <span style='background :lightgreen' >3. What is Self-Attention?</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Self-attention is a mechansim that allows a token to look at all other tokens in the sequence and decide, <b>"who to listen to"</b> and <b>"how much to listen to"</b> and then combine that information.</h3>


<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The output of  Self-attention block are contextual representation of each token, that capture relationships between words beyond just their standalone meanings and positional information.</h3>

<div style="text-align:center;">
    <img src="../images/self-attention-007.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>

<br><br><br><br>
# <span style='background :lightgreen' >4. How Self-Attention Work?</span>

<h1 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Self-attention lets each token decide “who to listen to,” how much to listen, and then combine that information.</h1>

<div style="text-align:center;">
    <img src="../images/self-attention-013.png"
         style="max-width:800px; width:100%; height:auto; display:inline-block;">
</div>

<img align=right src="../images/self-attention-14.png" width="600">

<h1 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Apple released a new iPhone model last week</h1>

<br><br><br><br><br>

<br><br><br><br>
<img align=right src="../images/self-attention-008.png" width="900">

# Step 1: Generate Q, K, and V Matrices
- The very first thing self-attention does is take each token's 512-dimensional input vector and project it into three completely different spaces; creating three new vectors called the Query vector ($\vec{q}$), the Key vector ($\vec{k}$) and the Value vector ($\vec{k}$)
- To do this we multiply the input matrix `X` by three separate learned weight matrices: `W_Q`, `W_K` and `W_V`,  and the result are three matrices: **`Q`**, **`K`** and **`V`**, each of shape **8 × 512**  (one row per token)
- Now what do these three vectors actually mean? Think of it like a search engine:
    - The **Query vector $\vec{q}$** is actually is a question, asked by a token from all other tokens including itself, about the information it need from other tokens to understand itself in context. You can think of it as a Google search query.
    - The **Key vector $\vec{k}$** of a token is a response from a tokens telling what information it can provide to other tokens. You can think of it as a web page title to the Google search query.
    - The **Value vector $\vec{v}$** of a token contains the real infromation. You can think of it as the web page content in response to the Google search query.
- These three weight matrices `W_Q`, `W_K` and `W_V` are learned during training — the model gradually learns the best way to project input vectors into meaningful query, key and value spaces — so that the attention mechanism learns to focus on the right relationships for the task at hand
- This entire step happens as one efficient matrix multiplication — all 8 tokens generate their Q, K and V vectors simultaneously in parallel

```
Query projection:         Q = X × W_q            (8 × 512) = (8 × 512) × (512 × 512)
Key projection:           K = X × W_k            (8 × 512) = (8 × 512) × (512 × 512)
Value projection:         V = X × W_v            (8 × 512) = (8 × 512) × (512 × 512)
```
- Where:
    - `X`: Input embeddings with positional encoding, shape `(seq_len, d_model)`. In our example it is `(8,512)`
    - `W_q, W_k, W_v`: Learnable weight matrices, each of shape `(d_model, d_k)`. In our example it is `(512,512)`
    - `d_k`: Dimension of queries/keys (typically d_model / num_heads)
- Finally we have three matrices **Q**, **K** and **V** each containing eight stacked $\vec{q}$, $\vec{k}$ and $\vec{v}$ vectors respectively, each corresponding to one input token.
- If the input embedding size is 512 dimension, then the size of **Q**, **K** and **V** matrices is 8 x 512. (where 8 is the count of input tokens in sequence)

<br><br><br><br><br><br><br><br><br><br>
<img align=right src="../images/self-attention-009.png" width="900">

# Step 2: (Generate Similarity Matrix)

- Now that we have our **Q** and **K** matrices, we compute a **similarity score** between every pair of tokens, that tells us how relevant each token is to every other token.
- For example, to calculate the similarity of a vector with all other vectors we take the **$\vec{q}$** vector of token **"apple"** and take its dot product with **$\vec{k}$** vectors of all other tokens to get 8 similarity scores.
- This entire step happens as one efficient matrix multiplication, i.e., - we multiply the Query matrix by the **transpose** of the Key matrix:
```
Scores  =  Q  ×  Kᵀ       shape: (8 × 8)
```
- The result is an **8 × 8 matrix**, where every row corresponds to one token, and every column corresponds to one token, and the value at position (i, j) tells us how similar **token i's query** is to **token j's key** (i.e., how much token i should attend to token j)
- A positive or high **dot product** between two vectors means the two vectors are pointing in a similar direction — meaning they are semantically related
- A negative or low **dot product** between two vectors means the two vectors are unrelated
- At this stage we have raw attention scores — but they have one problem — their values can be very large or very small depending on the dimensionality of the vectors — and this causes problems in the next step

<br><br><br><br><br> <br><br><br><br><br><br><br><br><br><br>
<img align=right src="../images/self-attention-010.png" width="600">

# Step 3: (Scale)
- The issue with the above similarity matrix is that, when we take dot product of two high dimensional vectors the resulting vector will have very high variance.
- Moreover, later we will be applying softmax function to each row of the score matrix and one of the characteristics of softmax function is that for very large and very small values, it maps them to probabilities close to 99% and 1%
- Solution is we scale by dividing the above matrix having similarity scores by the square root of dimension of the model, i.e., d_model
- Right now since we are not considering multi-headed self-attention, so `d_model` is the dimenstion of the $\vec{q}$ or $\vec{k}$ vectors, i.e. 512 in this example (for multi-headed self-attention, it is typically `d_model / num_heads`)

<br><br><br><br><br><br><br><br><br><br><br><br><br><br><br>
<img align=right src="../images/self-attention-011.png" width="900">

# Step 4: (Apply Softmax)
- Now we apply **Softmax row by row** to the scaled score matrix
- After Softmax — every row of our 8 × 8 matrix now contains 8 values/probabilities that are all positive and sum to exactly 1.0
- Each value tells us — **how much should this token attend to that token** — expressed as a percentage of attention
- These are the **attention weights** — and they are the heart of the self-attention mechanism — they encode the model's understanding of which tokens are most relevant to each other in this specific sentence
- Softmax takes exponent of each value and divides it with sum of all exponent across all values in that row using:

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j=1}^{n} e^{x_j}}$$

- Where:
    - $x_i$ = the score for position $i$ (one value from the scaled attention score row)
    - $e^{x_i}$ = Euler's number $e \approx 2.718$ raised to the power of $x_i$ — this ensures all values become positive
    - $\sum_{j=1}^{n} e^{x_j}$ = sum of all exponentiated values in that row; this normalises the values so they sum to exactly 1
    - $n$ = number of tokens in the sequence

<br><br><br><br><br><br><br><br><br><br><br><br><br><br><br>


# Step 5: (Compute Final Context-Aware Embeddings)

<img align=right src="../images/self-attention-012.png" width="1100">

- Now that we have the attention weights from Step 4, which is a matrix where every row sums to 1.0 and tells us exactly how much each token should attend to every other token.
- We use these weights to compute the final context-aware embedding for each token by multiplying the **attention weight matrix** with the **Value matrix V**:

$$\text{Output} = \text{Attention Weights} \times \mathbf{V}$$

$$\text{Output} = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right) \times \mathbf{V}$$

- The above multiplication actually takes a weighted sum of ALL value vectors $\vec{v}$ across the entire sequence
- The final output is a matrix of shape **(8 × 512)** — one 512-dimensional **context-aware embedding per token** (exactly the same shape as our input). Each vector now carries not just the meaning of its own token but a carefully weighted blend of information from every other token in the sequence.
- This is the entire point of self-attention — the output embeddings are no longer static lookup vectors — they are **dynamically constructed representations** that reflect what each token means specifically in the context of this particular sentence.
<br><br><br><br><br><br><br><br><br><br>

# <span style='background :lightgreen' >5. What is Masked (Causal) Self Attention?</span>
- The masked self-attention sub-component that exist inside the decoder layer, is quite similar to the self-attention inside the encoder layer (just discussed).
- The simple self-attention is bi-directional and is allowed to attend to all positions in the input sequence simultaneously.
- The masked self attention is uni-directional and is only allowed to attend to earlier positions in the output sequence.
- Language models predict **next token**, so to avoid cheating the models must not attend to future tokens

<h3 align="left"><div class="alert alert-success" color=magenta style="margin: 20px">Predict next word in the sentence: <b>"Ask not what your country can do for ......"</b></h3>

### Steps for Masked Self-Attention

**Masked self-attention** follows the exact same five steps with one single modification inserted between Step 3 and Step 4
```
Step 1 — Generate Q, K, V matrices          ← identical to simple self-attention  
Step 2 — Compute similarity scores  QKᵀ     ← identical to simple self-attention  
Step 3 — Scale by √d_k                      ← identical to simple self-attention  
           ↓
```
<span style="color:red; font-weight:bold;">ADD CAUSAL MASK</span>  
```
           ↓  
Step 4 — Apply Softmax row-wise             ← masked positions become exactly zero  
Step 5 — Multiply by V                      ← identical to simple self-attention
```
<br><br><br>
### Detailed Step-by-Step Masked Self-Attention

- **Step 1:** Generate Q, K, V by multiplying input X with learned weight matrices
- **Step 2:** Compute QKᵀ to get raw similarity scores between all token pairs
- **Step 3:** Divide by √d_k to scale the scores to a reasonable range

<div style="text-align:center;">
    <img src="../images/self-attention-15.png"
         style="max-width:1000px; width:100%; height:auto; display:inline-block;">
</div>

- **Step 4:** Apply Softmax row-wise to convert scores to attention weights summing to 1.0
    - In masked self-attention, we add a mask to the resulting matrix of step 3 before taking the sfotmax (which prevents the model from attending to future tokens)
    - Since transformer's decoder component is auto-regressive during inference time, ie.e., generating one token at a time, so it should not be allowed to see the future tokens
    - How masking works:
        - Apply an upper-triangular mask with `−∞`
        - Add mask to attention scores
        - Softmax turns masked positions into zero
        - Result: Token *i* can attend only to tokens ≤ *i*


<div style="text-align:center;">
    <img src="../images/self-attention-16.png"
         style="max-width:1200px; width:100%; height:auto; display:inline-block;">
</div>


- **Step 5:** Compute Final Context-Aware Embeddings
  
<div style="text-align:center;">
    <img src="../images/self-attention-17.png"
         style="max-width:1000px; width:100%; height:auto; display:inline-block;">
</div>

<br><br><br><br><br><br><br><br>

# <span style='background :lightgreen' >6. What is Cross Self Attention?</span>

<img align=right src="../images/self-attention-18.png" width="900">

- Cross Self-Attention exist inside the Decoder layers and is there in Encoder-Decoder Transformer architectures like BART and T5
- It acts like a bridge between the encoder and the decoder layers.
- The input to the cross-attention is output of masked self-attention, which is a rich representation of what has been generated so far.
- The decoder however, has not yet looked at the input sentence at all, which is required for tasks like translation and summarization.
- The cross-attention component fixes this:
    - The **Query comes from the decoder** asking the question "what information do I need from the input sentence right now?"
    - The **Key comes from the encoder** representing what each input token has to offer
    - The **Value comes from the encoder** containing rich contextual information from the encoder output

**Cross self-attention** follows the exact same five steps with one single modification, i.e., the K anv V came from encoder
```
Step 1 — Generate Q, K, V:                  ← THE ONLY DIFFERENT STEP
           Q  =  decoder_output  ×  W_Q     (from decoder)
           K  =  encoder_output  ×  W_K     (from encoder)
           V  =  encoder_output  ×  W_V     (from encoder)

Step 2 — Compute similarity scores:
           Scores  =  Q  ×  Kᵀ             ← identical to before

Step 3 — Scale:
           Scaled  =  Scores / √d_k        ← identical to before

Step 4 — Apply Softmax:
           Weights  =  softmax(Scaled)     ← NO mask added — full attention
                                              to all encoder positions

Step 5 — Combine with V:
           Output  =  Weights  ×  V        ← identical to before
```


<br><br><br><br><br><br>
# <span style='background :lightgreen' >7. What is Multi-Headed Self Attention?</span>

<div style="text-align:center;">
    <img src="../images/multi-headed-self-attn-1.png"
         style="max-width:1700px; width:100%; height:auto; display:inline-block;">
</div>


<div style="text-align:center;">
    <img src="../images/multi-headed-self-attn-2.png"
         style="max-width:800px; width:100%; height:auto; display:inline-block;">
</div>

# <span style='background :lightgreen' >8. Layer Normalization, Residual Connections and Feed Forward Neural Network</span>

<div style="text-align:center;">
    <img src="../images/dense-nn1.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

<br><br><br><br><br><br>

# <span style='background :lightgreen' >9. From 6th Decoder Output to Token Prediction</span>

<h1 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Predict next word in the sentence: <b>"Ask not what your country can do for ......</b></h1>

<img align=right src="../images/logits-prob.png" width="1000">

- Let us now trace exactly what happens inside the transformer after the input sentence has passed through all six decoder layers.
- Our input sentence is the famous JFK quote — **"Ask not what your country can do for..."** — and the model's job is to predict the missing ninth word.
- In this example, the output of the last decoder layer will be eight vectors enrished with contextual information along with stored factual knowldege.
- We only take **the very last vector**, i.e.,  for the token **"for"**, which has already attended to all previous tokens through the masked self-attention mechanism. So its vector already contains everything the model knows about the entire sentence up to this point. It is the most information-rich summary of the full context and it is the only one we need to predict the next word. 
- This single 512-dimensional vector now enters the **Language Model Head** — or LM Head for short. The LM Head is simply a single linear layer with no activation function: `nn.Linear(512, 37000)`. It takes our 512-dimensional vector living in **model space** and projects it into a 37,000-dimensional vector living in **vocabulary space** — one value per token in the entire vocabulary.
```
input:   (1 × 512)    ← model space
output:  (1 × 37000)  ← vocabulary space
```
- Each value in this output vector represents a raw score called logits, that can be any number from negative infinity to positive infinity. A high logit for a particular token means the model thinks that token is a likely next word. A low or negative logit means the model thinks it is unlikely.
- These raw logits are hard to interpret directly because they are unbounded. So we apply **Softmax** to convert them into proper probabilities
- After Softmax we now have 37,000 probability values where every single value is between 0 and 1 and all 37,000 values sum to exactly 1.0.
- Each value now tells us **"the probability that this vocabulary token is the correct next word?"**. Most of these 37,000 probabilities will be extremely small — close to zero. Only a handful of tokens will receive meaningful probability mass. The model has essentially distributed its confidence across the entire vocabulary.
- Now we simply find the index with the **highest probability** using argmax, which picks token ID corresponding to the word **"you"**.

# <span style='background :lightgreen' >10. Visualization of Transformer Architecture</span> 

## Visualizing Self-Attention:
#### Transformer Explainer: https://poloclub.github.io/transformer-explainer/
#### Tensor2Tensor Google Colab Notebook: https://colab.research.google.com/github/tensorflow/tensor2tensor/blob/master/tensor2tensor/notebooks/hello_t2t.ipynb#scrollTo=OJKU36QAfqOC
#### Online tool: https://huggingface.co/spaces/exbert-project/exbert
#### Online tool: https://projector.tensorflow.org/


<div style="text-align:center;">
    <img src="../images/tfmr-explainer.png"
         style="max-width:2000px; width:100%; height:auto; display:inline-block;">
</div>


<br><br><br><br>
# <span style='background :lightgreen' >11. Example Questions to Check your Understanding about Transformer Architecture</span>

**Question 1**

You have a sequence of **3 tokens** with embedding dimension **d_model = 4**.
The input embedding matrix (with positional encoding already added) is:
```
X = [ [1.0,  0.0,  1.0,  0.0],    ← token 1
      [0.0,  1.0,  0.0,  1.0],    ← token 2
      [1.0,  1.0,  0.0,  0.0] ]   ← token 3

Shape of X: (3 × 4)
```

The weight matrices are (d_k = 4):
```
W_q = W_k = W_v = Identity matrix (4 × 4)
      [[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 1]]
```

- **(a)** Compute the Q, K, and V matrices.
          What is the shape of each matrix?

- **(b)** Compute the raw similarity matrix `S = Q × Kᵀ`.
          Show the full (3 × 3) matrix and explain what
          entry S[0][1] means conceptually.

- **(c)** Scale the similarity matrix by dividing by √d_k.
          What is the value of √d_k here?
          Show the full scaled matrix.

- **(d)** Apply Softmax **row-wise** to the scaled matrix.
          Use the formula:
```
softmax(x_i) = exp(x_i) / Σ exp(x_j)
```
          Show the full (3 × 3) attention weight matrix.
          Verify that each row sums to 1.0.

- **(e)** Compute the final output matrix `Z = Attention_weights × V`.
          What is the shape of Z?
          What does each row of Z represent?

<details>
<summary>💡 <b>Click to reveal answer</b></summary>
```
Given:
  seq_len = 3,  d_model = d_k = d_v = 4
  W_q = W_k = W_v = Identity matrix

(a) Q, K, V matrices:
    Since W_q = W_k = W_v = Identity:
      Q = X × I = X
      K = X × I = X
      V = X × I = X

    Q = K = V = [ [1.0, 0.0, 1.0, 0.0],
                  [0.0, 1.0, 0.0, 1.0],
                  [1.0, 1.0, 0.0, 0.0] ]

    Shape of each: (3 × 4)  ← (seq_len × d_k)

(b) Raw similarity matrix S = Q × Kᵀ:
    Kᵀ shape = (4 × 3), so S = (3×4) × (4×3) = (3×3)

    S[i][j] = dot product of row i of Q with row j of K

    S[0][0] = [1,0,1,0]·[1,0,1,0] = 1+0+1+0 = 2
    S[0][1] = [1,0,1,0]·[0,1,0,1] = 0+0+0+0 = 0
    S[0][2] = [1,0,1,0]·[1,1,0,0] = 1+0+0+0 = 1
    S[1][0] = [0,1,0,1]·[1,0,1,0] = 0+0+0+0 = 0
    S[1][1] = [0,1,0,1]·[0,1,0,1] = 0+1+0+1 = 2
    S[1][2] = [0,1,0,1]·[1,1,0,0] = 0+1+0+0 = 1
    S[2][0] = [1,1,0,0]·[1,0,1,0] = 1+0+0+0 = 1
    S[2][1] = [1,1,0,0]·[0,1,0,1] = 0+1+0+0 = 1
    S[2][2] = [1,1,0,0]·[1,1,0,0] = 1+1+0+0 = 2

    S = [ [2, 0, 1],
          [0, 2, 1],
          [1, 1, 2] ]

    S[0][1] = 0 means: token 1's query has ZERO similarity
    with token 2's key — token 1 finds token 2 completely
    irrelevant and will assign it very low attention weight

(c) Scaled similarity matrix S_scaled = S / √d_k:
    √d_k = √4 = 2.0

    S_scaled = S / 2.0

    S_scaled = [ [1.000, 0.000, 0.500],
                 [0.000, 1.000, 0.500],
                 [0.500, 0.500, 1.000] ]

(d) Row-wise Softmax to get attention weights A:

    Row 0: softmax([1.000, 0.000, 0.500])
      exp(1.000) = 2.7183
      exp(0.000) = 1.0000
      exp(0.500) = 1.6487
      sum        = 5.3670
      A[0] = [2.7183/5.3670, 1.0000/5.3670, 1.6487/5.3670]
           = [0.5064,        0.1863,         0.3072]
      sum check: 0.5064 + 0.1863 + 0.3072 = 0.9999 ≈ 1.0  ✅

    Row 1: softmax([0.000, 1.000, 0.500])
      exp(0.000) = 1.0000
      exp(1.000) = 2.7183
      exp(0.500) = 1.6487
      sum        = 5.3670
      A[1] = [1.0000/5.3670, 2.7183/5.3670, 1.6487/5.3670]
           = [0.1863,        0.5064,         0.3072]
      sum check: 0.1863 + 0.5064 + 0.3072 = 0.9999 ≈ 1.0  ✅

    Row 2: softmax([0.500, 0.500, 1.000])
      exp(0.500) = 1.6487
      exp(0.500) = 1.6487
      exp(1.000) = 2.7183
      sum        = 5.0157
      A[2] = [1.6487/5.0157, 1.6487/5.0157, 2.7183/5.0157]
           = [0.3287,        0.3287,         0.5426]

      NOTE: rows 0 and 1 have the same exp values but
      in different positions — this is correct since
      S_scaled[0] and S_scaled[1] are mirrors of each other

      sum check: 0.3287 + 0.3287 + 0.5426 = 1.0000  ✅

    A = [ [0.5064, 0.1863, 0.3072],
          [0.1863, 0.5064, 0.3072],
          [0.3287, 0.3287, 0.5426] ]

(e) Output Z = A × V:
    Shape: (3×3) × (3×4) = (3×4)  ← same as input X ✅

    Z[0] = 0.5064×[1,0,1,0] + 0.1863×[0,1,0,1] + 0.3072×[1,1,0,0]
         = [0.5064, 0,      0.5064, 0     ]
         + [0,      0.1863, 0,      0.1863]
         + [0.3072, 0.3072, 0,      0     ]
         = [0.8136, 0.4935, 0.5064, 0.1863]

    Z[1] = 0.1863×[1,0,1,0] + 0.5064×[0,1,0,1] + 0.3072×[1,1,0,0]
         = [0.1863, 0,      0.1863, 0     ]
         + [0,      0.5064, 0,      0.5064]
         + [0.3072, 0.3072, 0,      0     ]
         = [0.4935, 0.8136, 0.1863, 0.5064]

    Z[2] = 0.3287×[1,0,1,0] + 0.3287×[0,1,0,1] + 0.5426×[1,1,0,0]
         = [0.3287, 0,      0.3287, 0     ]
         + [0,      0.3287, 0,      0.3287]
         + [0.5426, 0.5426, 0,      0     ]
         = [0.8713, 0.8713, 0.3287, 0.3287]

    Z = [ [0.8136, 0.4935, 0.5064, 0.1863],   ← contextual embedding token 1
          [0.4935, 0.8136, 0.1863, 0.5064],   ← contextual embedding token 2
          [0.8713, 0.8713, 0.3287, 0.3287] ]  ← contextual embedding token 3

    Shape of Z: (3 × 4)  ✅

    Each row of Z is the CONTEXTUAL EMBEDDING for that token —
    a weighted blend of all value vectors, where the weights
    were determined by how relevant each token's key was to
    that token's query. Unlike the original X where each token
    only contained its own information, each row of Z now
    contains information gathered from ALL tokens in the
    sequence, weighted by attention scores.
```
</details>

**Question 2**

You are given the following pre-computed **attention weight matrix**
for a 4-token sequence. Each row represents one token's attention
distribution over all tokens in the sequence:
```
Tokens:  ["The",  "cat",  "sat",  "down"]
Indices:    0        1       2       3

Attention weight matrix (already after Softmax):

          "The"  "cat"  "sat"  "down"
"The"  [ [0.70,  0.20,  0.05,  0.05],   ← row 0
"cat"    [0.10,  0.60,  0.20,  0.10],   ← row 1
"sat"    [0.05,  0.40,  0.40,  0.15],   ← row 2
"down" ] [0.05,  0.15,  0.50,  0.30] ]  ← row 3
```

The Value matrix is:
```
V = [ [1.0,  0.0],    ← value vector for "The"
      [0.0,  1.0],    ← value vector for "cat"
      [1.0,  1.0],    ← value vector for "sat"
      [0.0,  0.5] ]   ← value vector for "down"

Shape: (4 × 2),   d_v = 2
```

- **(a)** Verify that the attention weight matrix is valid.
          What two properties must every row satisfy?
          Check both properties for row 2.

- **(b)** Looking at the attention weight matrix, which token
          does "sat" (row 2) attend to most strongly?
          What does this tell us about the contextual
          relationship the model has learned?

- **(c)** Compute the contextual output vector for the token
          **"sat"** (row 2) using:
```
Z[2] = Σ attention_weight[2][j] × V[j]   for j = 0, 1, 2, 3
```
          Show all intermediate steps.

- **(d)** Compute the contextual output vector for the token
          **"cat"** (row 1).

- **(e)** Compare Z[1] ("cat") and Z[2] ("sat").
          Both tokens exist in the same sentence.
          Why are their output vectors different even though
          they use the same Value matrix?

- **(f)** A student claims: *"Since all rows use the same
          Value matrix V, all output vectors Z must be identical."*
          Is the student correct? Explain precisely why or why not.

<details>
<summary>💡 <b>Click to reveal answer</b></summary>
```
Given:
  4 tokens: ["The", "cat", "sat", "down"]
  Attention weights A: (4 × 4) — already post-Softmax
  Value matrix V: (4 × 2),  d_v = 2

(a) Two properties every valid attention weight row must satisfy:
    Property 1: All values must be NON-NEGATIVE (≥ 0)
                — they are probabilities, negative probabilities
                  are meaningless
    Property 2: All values in each row must SUM TO 1.0
                — they form a probability distribution over tokens

    Checking row 2 = [0.05, 0.40, 0.40, 0.15]:
      Non-negative check: 0.05 ≥ 0 ✅  0.40 ≥ 0 ✅
                          0.40 ≥ 0 ✅  0.15 ≥ 0 ✅
      Sum check: 0.05 + 0.40 + 0.40 + 0.15 = 1.00  ✅
      Row 2 is a valid attention distribution  ✅

(b) "sat" (row 2) attention weights = [0.05, 0.40, 0.40, 0.15]
    Highest weights: "cat" (0.40) and "sat" itself (0.40) — tied

    This tells us the model has learned that "sat" is most
    strongly related to "cat" (its grammatical subject) and
    to itself. This makes linguistic sense: to understand what
    "sat" means in context, the model focuses on WHO is doing
    the sitting ("cat") and the action itself ("sat").
    "down" gets 0.15 (some relevance as a modifier) while
    "The" gets only 0.05 (the article is largely irrelevant
    to the verb "sat")

(c) Contextual output for "sat" — Z[2]:
    Z[2] = 0.05×V[0] + 0.40×V[1] + 0.40×V[2] + 0.15×V[3]

    0.05 × [1.0, 0.0] = [0.050, 0.000]
    0.40 × [0.0, 1.0] = [0.000, 0.400]
    0.40 × [1.0, 1.0] = [0.400, 0.400]
    0.15 × [0.0, 0.5] = [0.000, 0.075]
    ─────────────────────────────────
    Z[2]              = [0.450, 0.875]

    Verification:
    dim 0: 0.050 + 0.000 + 0.400 + 0.000 = 0.450  ✅
    dim 1: 0.000 + 0.400 + 0.400 + 0.075 = 0.875  ✅

(d) Contextual output for "cat" — Z[1]:
    Z[1] = 0.10×V[0] + 0.60×V[1] + 0.20×V[2] + 0.10×V[3]

    0.10 × [1.0, 0.0] = [0.100, 0.000]
    0.60 × [0.0, 1.0] = [0.000, 0.600]
    0.20 × [1.0, 1.0] = [0.200, 0.200]
    0.10 × [0.0, 0.5] = [0.000, 0.050]
    ─────────────────────────────────
    Z[1]              = [0.300, 0.850]

    Verification:
    dim 0: 0.100 + 0.000 + 0.200 + 0.000 = 0.300  ✅
    dim 1: 0.000 + 0.600 + 0.200 + 0.050 = 0.850  ✅

(e) Comparison of Z[1] and Z[2]:
    Z[1] ("cat")  = [0.300, 0.850]
    Z[2] ("sat")  = [0.450, 0.875]

    They are DIFFERENT because their attention weight ROWS
    are different:
      "cat" row  = [0.10, 0.60, 0.20, 0.10]
                   → heavily weights itself (0.60)
      "sat" row  = [0.05, 0.40, 0.40, 0.15]
                   → splits attention between "cat" and itself

    Even though both use the same V matrix, each token blends
    the value vectors with DIFFERENT weights — producing a
    unique contextual embedding that reflects each token's
    specific role and relationships in the sentence.
    This is the core power of self-attention: same V, different
    attention patterns → different contextual representations.

(f) The student is INCORRECT.
    The output vectors Z are computed as:
      Z[i] = Σ A[i][j] × V[j]

    While V is shared across all tokens, each token i has its
    OWN unique attention weight ROW A[i] — determined by how
    that token's Query matched every token's Key.
    Different attention rows → different weighted combinations
    of V → different output vectors.

    The outputs would only be identical if every row of A were
    identical — which would mean every token attends to all
    other tokens in exactly the same way, making attention
    completely useless.

    In practice, attention rows are always different because
    each token has a different query, capturing different
    contextual relationships — which is the entire purpose
    of self-attention.
```
</details>


**Question 3**
- **(a)** In self-attention, every token produces three vectors Q, K, V.
        In ONE sentence explain what each of these three vectors represents
        conceptually (what role does each play in attention?)
- **(b)** Why do we SCALE the similarity scores by dividing by √d_k
        before applying Softmax? What problem does scaling solve?
- **(c)** The attention formula is:
```
Attention(Q, K, V) = Softmax( Q × Kᵀ / √d_k ) × V
```
A student says: *"The Softmax output tells us the actual meaning
of each token."* Is the student correct? What does the Softmax
output actually represent?

- **(d)** What is the key difference between:
    - **Masked Self-Attention** (used in the decoder)
    - **Regular Self-Attention** (used in the encoder)

<details>
<summary>💡 Click to reveal answer</summary>

```
(a) Role of Q, K, V:

    Q (Query)  → "What am I looking for?"
                 Each token asks a question about what context it needs
                 Token "it" asks: "which noun does 'it' refer to?"

    K (Key)    → "What information do I have to offer?"
                 Each token advertises its content to other tokens
                 Token "cat" says: "I am a noun, an animal, a subject"

    V (Value)  → "What information do I actually pass along?"
                 The actual content a token shares when attended to
                 Token "cat" passes its rich semantic representation

    Attention = match Q against all Ks to find relevance,
                then use that relevance to weight the Vs

(b) Why scale by √d_k:
    Q × Kᵀ produces dot products that grow in MAGNITUDE
    as d_k increases (more dimensions = larger dot products)
    Large dot products → extreme Softmax inputs → Softmax output
    becomes very close to 0 or 1 (near one-hot distribution)
    → Gradients become near zero → vanishing gradient problem!

    Dividing by √d_k keeps dot product magnitudes in a reasonable
    range regardless of d_k → stable Softmax gradients → stable training
    Example: d_k=64 → divide by √64=8  │  d_k=512 → divide by √512≈22.6

(c) The student is INCORRECT.
    The Softmax output is an ATTENTION WEIGHT DISTRIBUTION —
    a set of probabilities (summing to 1.0) that tells the model:
    "pay THIS MUCH attention to each token when computing my output"

    It represents RELEVANCE SCORES between tokens, not meaning.
    The actual meaning/content comes from the V (Value) vectors —
    the Softmax weights determine HOW MUCH of each V to include
    in the final output via the weighted sum.

(d) Masked Self-Attention vs Regular Self-Attention:

    Regular Self-Attention (Encoder):
      Every token can attend to EVERY other token in the sequence
      Token 3 can see tokens 1, 2, 4, 5, 6... (past AND future)
      Used in encoder — full context is available (e.g., translation input)

    Masked Self-Attention (Decoder):
      Each token can only attend to PREVIOUS tokens (left context only)
      Token 3 can see tokens 1, 2 but NOT tokens 4, 5, 6...
      Future positions are masked with -∞ before Softmax → become 0
      Used in decoder — prevents the model from "cheating" by
      looking at future tokens it is supposed to GENERATE
```
</details>


**Q4. Multi-Head Self-Attention**

The original transformer uses **h = 8 attention heads** with
**d_model = 512**.

**(a)** What is the dimension of Q, K, V per head (d_k and d_v)?
        Show the calculation.

**(b)** A student says:
        *"Multi-head attention is just running 8 separate attention
        operations and then averaging the results."*
        Identify TWO mistakes in this statement.

**(c)** What is the shape of the output from ONE attention head?
        (sequence length = 10 tokens)

**(d)** After running all 8 heads, the outputs are concatenated.
        What is the shape of the concatenated output?

**(e)** A final weight matrix W_O is applied after concatenation.
        What must the shape of W_O be, and what does it do?

**(f)** Why is multi-head attention better than single-head attention
        with d_k = 512? What can multiple heads capture that one
        head cannot?

<details>
<summary>💡 Click to reveal answer</summary>

```
Given: h=8 heads, d_model=512, sequence_length=10

(a) d_k = d_v = d_model / h
              = 512 / 8
              = 64

    Each head works with 64-dimensional Q, K, V vectors
    (instead of the full 512 dimensions)

(b) Two mistakes in the student's statement:

    Mistake 1 — "averaging":
    The outputs are CONCATENATED, not averaged.
    concat([head1, head2, ..., head8]) → shape (seq × 512)
    Averaging would lose information from individual heads

    Mistake 2 — "just running 8 separate operations":
    Each head uses DIFFERENT learned weight matrices
    (W_Q_i, W_K_i, W_V_i for head i)
    Each head learns to attend to DIFFERENT types of relationships
    They are not identical operations — they specialize!

(c) Shape of ONE attention head output:
    Each head produces: (sequence_length × d_v)
                      = (10 × 64)

(d) Shape after concatenating all 8 heads:
    = (sequence_length × (h × d_v))
    = (10 × (8 × 64))
    = (10 × 512)

(e) Shape of W_O:
    Input to W_O  = (10 × 512)  →  W_O shape = (512 × 512)
    Output        = (10 × 512)  →  same as d_model

    What W_O does:
    Projects the concatenated multi-head output back into d_model
    space — mixes and combines the information from all 8 heads
    into one unified representation per token

(f) Why multi-head > single-head (d_k=512):

    Single head with d_k=512:
    One set of Q,K,V → learns ONE type of attention relationship
    Can only focus on ONE aspect of context at a time

    8 heads with d_k=64 each:
    Head 1 might learn: syntactic relationships (subject-verb)
    Head 2 might learn: coreference (pronoun → noun)
    Head 3 might learn: local context (adjacent words)
    Head 4 might learn: semantic similarity
    Head 5-8 might learn: other linguistic patterns

    Multiple heads allow the model to simultaneously attend to
    DIFFERENT aspects of the input from DIFFERENT representation
    subspaces — richer and more expressive than one big head
    Total parameters are the same (8×64 = 512) but much more powerful!
```
</details>


**Q5. Masked Self-Attention**

You are in the decoder. The sequence so far is **3 tokens**.
The scaled similarity matrix before masking is:

```
         Token1  Token2  Token3
Token1 [  2.0     1.0     0.8  ]
Token2 [  1.5     2.5     1.2  ]
Token3 [  0.9     1.8     2.2  ]
```

**(a)** Apply the decoder mask. Write the masked matrix.
        *(Hint: future positions are replaced with -∞)*

**(b)** Apply Softmax row-wise to the masked matrix.
        Show the full calculation for Row 2 (Token 2) only.
        *(Use e^(-∞) = 0)*

**(c)** For Row 3 (Token 3), without full calculation, write what
        the Softmax output will look like qualitatively
        (which positions will be zero and which will be non-zero?)

**(d)** A student asks: *"Why do we use -∞ for masking instead
        of just putting 0?"* Give a precise explanation.

**(e)** Cross-attention in the decoder uses:
        - Q from the DECODER
        - K and V from the ENCODER

        Why? What would happen if K and V also came from the decoder?

<details>
<summary>💡 Click to reveal answer</summary>

```
(a) Decoder mask — mask all positions ABOVE the diagonal with -∞:
    (Token i cannot attend to Token j if j > i)

             Token1  Token2   Token3
    Token1 [  2.0     -∞       -∞   ]   ← can only see itself
    Token2 [  1.5     2.5      -∞   ]   ← can see tokens 1,2
    Token3 [  0.9     1.8      2.2  ]   ← can see all 3 tokens

(b) Softmax for Row 2 (Token 2): [1.5, 2.5, -∞]

    Step 1 — exponentiate:
      e^1.5  = 4.4817
      e^2.5  = 12.1825
      e^(-∞) = 0.0000   ← masked position contributes nothing

    Step 2 — sum:
      sum = 4.4817 + 12.1825 + 0.0000 = 16.6642

    Step 3 — divide:
      A[2,1] = 4.4817  / 16.6642 = 0.2690
      A[2,2] = 12.1825 / 16.6642 = 0.7310
      A[2,3] = 0.0000  / 16.6642 = 0.0000  ← zero attention to future

    Row 2 of A = [0.2690, 0.7310, 0.0000]
    ✅ Check: 0.2690 + 0.7310 + 0.0000 = 1.0000 ✅

    Token 2 attends 73% to itself and 27% to Token 1
    and ZERO to Token 3 (future token — correctly masked!) ✅

(c) Row 3 (Token 3) qualitative Softmax output:
    Input row = [0.9, 1.8, 2.2]  ← no masking needed for last token!
    ALL three positions are visible to Token 3 (no future tokens)

    Softmax output: [small, medium, largest]
    All three values will be NON-ZERO since no position is masked
    Token 3 attends most to itself (2.2 is highest)
    followed by Token 2 (1.8) then Token 1 (0.9)

(d) Why -∞ instead of 0:

    If we put 0 in masked positions:
      Softmax([2.0, 0, 0]) → [e^2/(e^2+e^0+e^0)] = [0.576, 0.212, 0.212]
      The masked positions get Softmax output 0.212 — NOT zero!
      The model still attends to future tokens with 21% weight each!
      Masking with 0 FAILS to prevent future token attention

    If we put -∞ in masked positions:
      Softmax([2.0, -∞, -∞]) → [e^2/(e^2+0+0)] = [1.0, 0.0, 0.0]
      e^(-∞) = 0 exactly → masked positions get EXACTLY zero weight
      The model attends to future tokens with 0% weight ✅

    -∞ is the mathematically correct way to force exactly zero
    attention weight after Softmax — it is the only value that
    guarantees complete masking regardless of the other scores

(e) Why Cross-Attention uses Q from decoder, K and V from encoder:

    Q from decoder = "What does my current decoder state need?"
    K from encoder = "What information does each encoder token offer?"
    V from encoder = "What content does each encoder token contain?"

    The decoder QUERIES the encoder output for relevant context.
    For translation: decoder generating "chat" (French) queries
    the encoder's representation of "cat" (English) — it needs
    to know WHAT the encoder understood about the source sentence

    If K and V also came from the decoder:
    The decoder would only attend to its own partial output
    (the tokens generated so far) — it would NEVER access
    the encoder's understanding of the source sentence!
    The model would have no connection to the input — it would
    just be predicting the next token from previous predictions
    with no access to what it is supposed to be translating/answering
```
</details>


**Q6. Attention Score Analysis — Putting It All Together**

You have a 3-token sequence: **"The cat sat"**
After computing scaled dot-product attention, the attention weight
matrix (after Softmax) for the ENCODER is:

```
            "The"   "cat"   "sat"
"The"    [  0.60    0.30    0.10  ]
"cat"    [  0.20    0.65    0.15  ]
"sat"    [  0.10    0.45    0.45  ]
```

**(a)** Verify that this is a valid attention weight matrix.
        What two properties must every row satisfy?

**(b)** Interpret the attention pattern for the word **"sat"**.
        Which token does "sat" attend to most? Why might this
        make linguistic sense?

**(c)** The Value vectors are:

```
V_The = [1.0, 0.0]
V_cat = [0.0, 1.0]
V_sat = [0.5, 0.5]
```

Compute the complete attention output matrix Z (3×2).
Show all three rows.

**(d)** Now suppose this were a DECODER with masked self-attention.
        Rewrite the attention weight matrix after masking is applied.
        *(Recompute Softmax for each row after masking — Row 1 only
        needs full calculation, state results for rows 2 and 3)*

**(e)** Compare the encoder output Z (from part c) with what the
        decoder output would be for "sat" after masking.
        What information does "sat" lose in the decoder vs encoder?
        Why is this acceptable in a decoder?

<details>
<summary>💡 Click to reveal answer</summary>

```
(a) Valid attention weight matrix properties:
    Property 1: Every value must be between 0 and 1 (non-negative)
    Property 2: Every ROW must sum to exactly 1.0 (probability dist)

    Check:
    Row 1: 0.60 + 0.30 + 0.10 = 1.00 ✅
    Row 2: 0.20 + 0.65 + 0.15 = 1.00 ✅
    Row 3: 0.10 + 0.45 + 0.45 = 1.00 ✅
    All values between 0 and 1 ✅  → Valid attention weight matrix

(b) "sat" attention pattern: [0.10, 0.45, 0.45]
    "sat" attends equally to "cat" (0.45) and itself (0.45)
    and very little to "The" (0.10)

    Linguistic interpretation:
    "sat" is a VERB — it needs to find its SUBJECT to understand
    the full meaning. "cat" is the subject of "sat" — the model
    has correctly learned that the verb should attend strongly
    to the noun that performs the action.
    Equal attention to itself (0.45) is also expected — a word
    always needs its own representation in the output.

(c) Z = A × V  (3×2 output matrix)

    Row 1 ("The"):  A = [0.60, 0.30, 0.10]
      dim0: 0.60×1.0 + 0.30×0.0 + 0.10×0.5 = 0.60 + 0.00 + 0.05 = 0.65
      dim1: 0.60×0.0 + 0.30×1.0 + 0.10×0.5 = 0.00 + 0.30 + 0.05 = 0.35
      Z_The = [0.65, 0.35]

    Row 2 ("cat"):  A = [0.20, 0.65, 0.15]
      dim0: 0.20×1.0 + 0.65×0.0 + 0.15×0.5 = 0.20 + 0.00 + 0.075 = 0.275
      dim1: 0.20×0.0 + 0.65×1.0 + 0.15×0.5 = 0.00 + 0.65 + 0.075 = 0.725
      Z_cat = [0.275, 0.725]

    Row 3 ("sat"):  A = [0.10, 0.45, 0.45]
      dim0: 0.10×1.0 + 0.45×0.0 + 0.45×0.5 = 0.10 + 0.00 + 0.225 = 0.325
      dim1: 0.10×0.0 + 0.45×1.0 + 0.45×0.5 = 0.00 + 0.45 + 0.225 = 0.675
      Z_sat = [0.325, 0.675]

    Complete output:
    Z = [[0.650, 0.350],    ← "The" output
         [0.275, 0.725],    ← "cat" output
         [0.325, 0.675]]    ← "sat" output

(d) Decoder masked attention:
    Original scaled scores (assume they produce the given weights
    before masking — work backwards or treat given matrix as scores)

    Using the given attention weights as pre-Softmax scores:

    Row 1 ("The") — masked: [0.60, -∞, -∞]
      e^0.60 = 1.8221,  e^-∞ = 0,  e^-∞ = 0
      sum = 1.8221
      A_masked[1] = [1.0000, 0.0000, 0.0000]
      ("The" can only attend to itself)

    Row 2 ("cat") — masked: [0.20, 0.65, -∞]
      e^0.20 = 1.2214,  e^0.65 = 1.9155,  e^-∞ = 0
      sum = 3.1369
      A_masked[2] = [0.3894, 0.6106, 0.0000]
      ("cat" attends to "The" and itself only)

    Row 3 ("sat") — no masking needed (last token sees all):
      A_masked[3] = [0.10, 0.45, 0.45]  ← unchanged

(e) Comparison — "sat" in encoder vs decoder:

    Encoder "sat" output: Z_sat = [0.325, 0.675]
    Uses full attention: [0.10, 0.45, 0.45]
    → "sat" has access to "The" (0.10), "cat" (0.45), itself (0.45)
    → Full bidirectional context → richest representation

    Decoder "sat" output: Z_sat = same [0.325, 0.675] here
    (since "sat" is last token — no future tokens to mask)
    BUT for earlier tokens like "cat":

    Encoder "cat": [0.275, 0.725] — attended to all 3 tokens
    Decoder "cat": attended to only "The" and itself
    → Cannot see "sat" which comes AFTER it in sequence
    → Less rich representation — missing future context

    Why this is acceptable in the decoder:
    The decoder generates text LEFT TO RIGHT at inference time —
    when generating token 2 ("cat"), token 3 ("sat") does NOT
    EXIST YET. The model cannot cheat by looking at words it
    has not generated yet. The masking enforces the autoregressive
    property — generate one token at a time using only the past.
    During training, masking simulates this inference condition
    so the model learns to predict without future information.
```
</details>


**Q7. Multi-Head Attention**

The original transformer uses h=8 heads and d_model=512.
Consider the following alternative designs:

```
Design A (original): h=8,  d_model=512, d_k=64
Design B           : h=1,  d_model=512, d_k=512
Design C           : h=16, d_model=512, d_k=32
Design D           : h=8,  d_model=512, d_k=128  ← student's proposal
```

**(a)** Calculate the total number of parameters in the Q projection
        matrices (W_Q for all heads combined) for each design.
        *(Each head has its own W_Q of shape d_model × d_k)*

**(b)** Design D has d_k=128 with h=8 heads.
        What is wrong with this design mathematically?
        Can it work? What constraint must d_k satisfy?

**(c)** Calculate the total FLOPs (floating point operations) for
        computing Q×Kᵀ for ONE head in Design A vs Design B,
        for a sequence of **100 tokens**.
        *(Matrix multiply of shape (n×d_k) × (d_k×n) costs 2×n×n×d_k FLOPs)*

**(d)** A researcher proposes Design C (16 heads, d_k=32).
        - How does it compare to Design A in total parameters?
        - What is the risk of using very small d_k=32 per head?
        - When might Design C be preferred over Design A?

**(e)** The total self-attention computation scales as O(n²×d) where
        n is sequence length and d is d_model.
        For a sequence of n=1000 tokens vs n=100 tokens (d_model=512):
        - How many times more computation does n=1000 require?
        - Why is this quadratic scaling a major problem for LLMs?
        - Name ONE modern technique that addresses this problem.

<details>
<summary>💡 Click to reveal answer</summary>

```
(a) Parameters in W_Q matrices (all heads combined):

    Each head has W_Q shape = (d_model × d_k)

    Design A (h=8,  d_k=64):
      Per head    = 512 × 64  = 32,768
      All 8 heads = 8 × 32,768 = 262,144 params

    Design B (h=1,  d_k=512):
      Per head    = 512 × 512 = 262,144
      1 head      = 262,144 params

    Design C (h=16, d_k=32):
      Per head    = 512 × 32  = 16,384
      All 16 heads = 16 × 16,384 = 262,144 params

    Design D (h=8,  d_k=128):
      Per head    = 512 × 128 = 65,536
      All 8 heads = 8 × 65,536 = 524,288 params

    Interesting: A, B, C all have IDENTICAL parameter counts (262,144)!
    Only Design D has more parameters.

(b) Problem with Design D (h=8, d_k=128):

    After concatenating all heads:
    Output shape = h × d_k = 8 × 128 = 1024

    But d_model = 512!
    The concatenated output (1024) does not match d_model (512)
    → W_O cannot project back to d_model correctly

    The constraint is:  h × d_k = d_model
    Design A: 8 × 64  = 512 ✅
    Design B: 1 × 512 = 512 ✅
    Design C: 16 × 32 = 512 ✅
    Design D: 8 × 128 = 1024 ≠ 512 ❌  INVALID!

    Design D cannot work without changing d_model to 1024
    or reducing h to 4 (4 × 128 = 512 ✅)

(c) FLOPs for Q×Kᵀ (sequence n=100):

    Formula: 2 × n × n × d_k FLOPs

    Design A (one head, d_k=64):
      = 2 × 100 × 100 × 64
      = 2 × 10,000 × 64
      = 1,280,000 FLOPs = 1.28M FLOPs per head
      For all 8 heads = 8 × 1.28M = 10.24M FLOPs

    Design B (one head, d_k=512):
      = 2 × 100 × 100 × 512
      = 2 × 10,000 × 512
      = 10,240,000 FLOPs = 10.24M FLOPs for the single head

    Design A total (8 heads): 10.24M FLOPs
    Design B total (1 head):  10.24M FLOPs
    → IDENTICAL total computation! ✅
    Multi-head attention has same compute as single-head
    but richer representations — no free lunch but no extra cost!

(d) Design C analysis (h=16, d_k=32):

    Total parameters: same as Design A (262,144) ← calculated above
    Compute: same as Design A (16 × 2×100×100×32 = 10.24M) ✅

    Risk of very small d_k=32:
    Each head only has 32 dimensions to work with
    → Very limited capacity per head to represent relationships
    → Each head may be too "narrow" to capture complex patterns
    → Diminishing returns — 16 very weak heads may underperform
       8 moderate heads

    When to prefer Design C:
    ✅ When you want more diverse specialization across more heads
    ✅ When tasks have many distinct linguistic relationship types
    ✅ When d_k=64 is empirically shown to be overkill for the task
    ✅ For smaller models where memory is critical

(e) Quadratic scaling O(n²×d):

    n=100 tokens:  compute ∝ 100²  × 512 = 5,120,000
    n=1000 tokens: compute ∝ 1000² × 512 = 512,000,000

    Ratio = 1000² / 100² = 1,000,000 / 10,000 = 100×

    n=1000 requires 100× more computation than n=100!
    (Doubling sequence length → 4× more computation)

    Why this is a major problem for LLMs:
    Modern LLMs need context windows of 8K, 32K, 128K tokens
    At 128K tokens: compute ∝ (128,000)² = 16.4 TRILLION operations
    just for ONE attention layer — completely impractical!
    Memory also scales as O(n²) for the attention matrix itself:
    128K × 128K attention matrix = 16.4 billion floats = ~32 TB!

    Modern techniques that address quadratic scaling:
    Any ONE of the following is acceptable:
    ✅ Flash Attention    — memory-efficient attention via tiling
    ✅ Sliding Window     — each token attends to local window only
    ✅ Sparse Attention   — attend to subset of positions
    ✅ Linear Attention   — approximates attention in O(n) time
    ✅ Ring Attention     — distributed attention across devices
    ✅ GQA / MQA          — Grouped/Multi-Query Attention (fewer K,V heads)
```
</details>


**Q8. Cross-Attention**

You are building a translation model (English → French).
The encoder processes the English sentence:
**"The cat sat"** (3 tokens)

The encoder output (after all encoder layers) is:

```
Encoder output H:
H_The = [1.0, 0.0]
H_cat = [0.0, 1.0]
H_sat = [0.5, 0.5]
```

The decoder has generated one token so far: **"Le"**
The decoder's current hidden state is:

```
Decoder state S_Le = [0.3, 0.7]
```

In cross-attention:
- **Q comes from the decoder** (projected from S_Le)
- **K and V come from the encoder** (projected from H)

Assume all projection matrices are identity (W_Q = W_K = W_V = I)
and d_k = 2.

**(a)** Write down the Q vector (from decoder) and the
        K, V matrix (from encoder)

**(b)** Compute the raw similarity scores between the decoder
        query Q and all 3 encoder keys K.
        *(3 dot products)*

**(c)** Scale the scores by √d_k and apply Softmax.
        Show full calculation.

**(d)** Compute the cross-attention output for "Le".
        What does this output vector represent?

**(e)** Interpret the result:
        - Which English word does "Le" attend to most?
        - Does this make linguistic sense for English → French translation?
        - What would it mean if "Le" attended equally to all 3 words?

**(f)** Now suppose the decoder has generated two tokens: "Le chat"
        In the DECODER'S masked self-attention (before cross-attention):
        - Draw the attention mask for ["Le", "chat"]
        - Can "Le" attend to "chat"? Can "chat" attend to "Le"?
        - After masked self-attention, does cross-attention change?
          (i.e., do K and V change when decoder generates more tokens?)

<details>
<summary>💡 Click to reveal answer</summary>

```
Given:
  Encoder output H = [[1.0, 0.0],   ← H_The
                       [0.0, 1.0],   ← H_cat
                       [0.5, 0.5]]   ← H_sat
  Decoder state S_Le = [0.3, 0.7]
  W_Q = W_K = W_V = Identity, d_k = 2

(a) Q vector (from decoder):
    Q_Le = S_Le × W_Q = [0.3, 0.7] × I = [0.3, 0.7]

    K matrix (from encoder — 3 key vectors):
    K_The = H_The × W_K = [1.0, 0.0]
    K_cat = H_cat × W_K = [0.0, 1.0]
    K_sat = H_sat × W_K = [0.5, 0.5]

    V matrix (from encoder — same as K here since W_V = I):
    V_The = [1.0, 0.0]
    V_cat = [0.0, 1.0]
    V_sat = [0.5, 0.5]

(b) Raw similarity scores (Q · K for each encoder position):

    score("The") = Q_Le · K_The = [0.3,0.7]·[1.0,0.0] = 0.3×1 + 0.7×0 = 0.30
    score("cat") = Q_Le · K_cat = [0.3,0.7]·[0.0,1.0] = 0.3×0 + 0.7×1 = 0.70
    score("sat") = Q_Le · K_sat = [0.3,0.7]·[0.5,0.5] = 0.3×0.5 + 0.7×0.5 = 0.50

    Raw scores = [0.30, 0.70, 0.50]

(c) Scale by √d_k = √2 = 1.4142:

    Scaled scores = [0.30/1.4142, 0.70/1.4142, 0.50/1.4142]
                  = [0.2121, 0.4950, 0.3536]

    Softmax:
      e^0.2121 = 1.2363
      e^0.4950 = 1.6407
      e^0.3536 = 1.4240
      sum      = 4.3010

      A_The = 1.2363 / 4.3010 = 0.2875
      A_cat = 1.6407 / 4.3010 = 0.3815
      A_sat = 1.4240 / 4.3010 = 0.3310

      Attention weights = [0.2875, 0.3815, 0.3310]
      ✅ Check: 0.2875 + 0.3815 + 0.3310 = 1.0000 ✅

(d) Cross-attention output for "Le":

    Z_Le = A · V
         = 0.2875×[1.0,0.0] + 0.3815×[0.0,1.0] + 0.3310×[0.5,0.5]

    dim0: 0.2875×1.0 + 0.3815×0.0 + 0.3310×0.5 = 0.2875 + 0 + 0.1655 = 0.4530
    dim1: 0.2875×0.0 + 0.3815×1.0 + 0.3310×0.5 = 0 + 0.3815 + 0.1655 = 0.5470

    Z_Le = [0.4530, 0.5470]

    What it represents:
    A context-aware representation for generating the French token "Le"
    that is a WEIGHTED MIX of all English encoder states — the decoder
    has looked at the entire English sentence and extracted the most
    relevant information to decide what French token to generate next

(e) Interpretation:

    Attention weights:
      "The" → 0.2875  (28.75%)
      "cat" → 0.3815  (38.15%) ← highest attention
      "sat" → 0.3310  (33.10%)

    "Le" attends MOST to "cat" (38.15%)
    This makes partial linguistic sense:
      "Le" in French means "The" (masculine article)
      → One might expect "Le" to attend most to "The"
      → However "cat" (le chat = masculine) also drives article choice
      → The model correctly picks up that "cat" (masculine noun)
         requires the masculine article "Le" — linguistically valid!

    If "Le" attended equally (0.333 each):
    The model would have NO preference for any source word
    → Cannot decide which French word to generate
    → The translation would be random and incoherent
    → Equal attention = model is confused = bad translation quality

(f) Masked self-attention for decoder ["Le", "chat"]:

    Attention mask:
              "Le"   "chat"
    "Le"   [  ✅      ❌  ]   ← "Le" cannot see "chat" (future)
    "chat" [  ✅      ✅  ]   ← "chat" can see "Le" and itself

    As a matrix with -∞ masking:
              "Le"   "chat"
    "Le"   [ score    -∞  ]
    "chat" [ score   score]

    Can "Le" attend to "chat"?   ❌ NO — "chat" is a future token
    Can "chat" attend to "Le"?   ✅ YES — "Le" is a past token

    Does cross-attention K and V change?
    ❌ NO — K and V in cross-attention ALWAYS come from the
    ENCODER output H which is computed ONCE and stays FIXED
    for the entire decoding process regardless of how many
    decoder tokens have been generated.

    Only Q changes (it comes from the current decoder state)
    → As the decoder generates more tokens, Q changes
    → But K and V from the encoder remain the same
    → This is the key mechanism: the encoder "stores" the full
       source sentence and the decoder repeatedly queries it
```
</details>